# 03 – Build Three-Tier Graph Artifact (Local)

Builds the heterogeneous graph NPZ that is then uploaded to Kaggle for HGT training.

**Prerequisites**
- `metadata.csv`           – from `01_extract_payload_from_pcap.ipynb`
- `payload_256.npy`        – from `01_extract_payload_from_pcap.ipynb`
- `student_embeddings.npy` – from `02_export_student_embeddings.ipynb`
- MITRE technique CSV + embeddings  – run `prepare_mitre_knowledge_base.py` first
  if not already present under `data/processed/mitre/`

**Output** → `data/processed/`
```
graph_artifact_3tier.npz                        ← main graph arrays
graph_artifact_3tier.meta.json                  ← graph metadata
graph_artifact_3tier_packet_semantic_x.npy      ← packet semantic feature matrix
graph_artifact_3tier_packet_semantic_x.npy.zst  ← compressed (after cell 5)
```

**Next step** → upload the three output files (or `.npy.zst` instead of `.npy`) to Kaggle
then run `../kaggle/02_train_hgt_flow_classifier.ipynb`.

In [ ]:
# ── Input paths ────────────────────────────────────────────────────────────────
METADATA_CSV                   = "data/interim/payload_dataset/metadata.csv"
PAYLOAD_NPY                    = "data/interim/payload_dataset/payload_256.npy"
STUDENT_EMBEDDING_NPY          = "data/processed/student_embeddings.npy"

MITRE_TECHNIQUES_CSV           = "data/processed/mitre/techniques.csv"
MITRE_TECHNIQUE_EMBEDDINGS_NPY = "data/processed/mitre/technique_embeddings.npy"
MITRE_TACTIC_EDGES_CSV         = "data/processed/mitre/technique_tactic_edges.csv"

# ── Output ─────────────────────────────────────────────────────────────────────
OUTPUT_NPZ = "data/processed/graph_artifact_3tier.npz"

# ── Graph construction hyperparameters ─────────────────────────────────────────
FLOW_TIMEOUT_SECONDS     = 30.0
MAX_PACKETS_PER_FLOW     = 20
SIMILARITY_THRESHOLD     = 0.82   # cosine cutoff for packet/flow → technique edges
PACKET_TOP_K             = 5      # top-K techniques per packet
FLOW_TOP_K               = 5      # top-K techniques per flow

DEVICE                   = "auto" # auto | cpu | cuda
SIM_BATCH_SIZE           = 50_000 # packets per batch for similarity computation

In [ ]:
# Pre-flight checks.
from pathlib import Path

required = {
    "metadata.csv":            METADATA_CSV,
    "payload_256.npy":         PAYLOAD_NPY,
    "student_embeddings.npy":  STUDENT_EMBEDDING_NPY,
    "techniques.csv":          MITRE_TECHNIQUES_CSV,
    "technique_embeddings.npy":MITRE_TECHNIQUE_EMBEDDINGS_NPY,
    "technique_tactic_edges.csv": MITRE_TACTIC_EDGES_CSV,
}

missing = []
for label, path in required.items():
    p = Path(path)
    if p.exists():
        print(f"[OK] {label:35s} {p.stat().st_size / 1e9:.2f} GB")
    else:
        print(f"[MISSING] {label} → {p}")
        missing.append(path)

if missing:
    raise FileNotFoundError(
        f"{len(missing)} required file(s) missing. See MISSING entries above."
    )
print("\nAll inputs found. Ready to build.")

In [ ]:
import subprocess, sys
from pathlib import Path

Path(OUTPUT_NPZ).parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m",
    "graphslm_ids.offline_path.preprocessing.build_three_tier_graph_artifact",
    "--metadata-csv",                     METADATA_CSV,
    "--payload-npy",                      PAYLOAD_NPY,
    "--student-embedding-npy",            STUDENT_EMBEDDING_NPY,
    "--mitre-techniques-csv",             MITRE_TECHNIQUES_CSV,
    "--mitre-technique-embeddings-npy",   MITRE_TECHNIQUE_EMBEDDINGS_NPY,
    "--mitre-technique-tactic-edges-csv", MITRE_TACTIC_EDGES_CSV,
    "--output-npz",                       OUTPUT_NPZ,
    "--flow-timeout-seconds",             str(FLOW_TIMEOUT_SECONDS),
    "--max-packets-per-flow",             str(MAX_PACKETS_PER_FLOW),
    "--similarity-threshold",            str(SIMILARITY_THRESHOLD),
    "--packet-top-k",                     str(PACKET_TOP_K),
    "--flow-top-k",                       str(FLOW_TOP_K),
    "--device",                           DEVICE,
    "--sim-batch-size",                   str(SIM_BATCH_SIZE),
]

print("$", " ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
# Verify outputs.
import json
import numpy as np
from pathlib import Path

npz_path     = Path(OUTPUT_NPZ)
meta_path    = npz_path.with_suffix(".meta.json")
pkt_sem_path = npz_path.with_name(npz_path.stem + "_packet_semantic_x.npy")

print(f"graph_artifact_3tier.npz               : {npz_path.stat().st_size / 1e9:.2f} GB")

if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print("\nGraph metadata:")
    for k, v in meta.items():
        if k not in ("node_count_detail", "created_at_utc"):
            print(f"  {k}: {v}")

if pkt_sem_path.exists():
    sem = np.load(pkt_sem_path, mmap_mode="r")
    print(f"\npacket_semantic_x.npy: {sem.shape}  "
          f"{pkt_sem_path.stat().st_size / 1e9:.2f} GB")

print("\nFiles ready for Kaggle upload:")
for p in [npz_path, meta_path, pkt_sem_path]:
    if p.exists():
        print(f"  {p}")

In [ ]:
# (Optional) Compress packet_semantic_x.npy with Zstandard before uploading.
# Reduces a ~55 GB file to ~20-25 GB. The Kaggle HGT notebook decompresses
# it automatically.
from pathlib import Path
import subprocess, sys

try:
    import zstandard as zstd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "zstandard"])
    import zstandard as zstd

src = Path(OUTPUT_NPZ).with_name(Path(OUTPUT_NPZ).stem + "_packet_semantic_x.npy")
dst = src.with_suffix(".npy.zst")

if not src.exists():
    print(f"packet_semantic_x.npy not found at {src} – skipping compression.")
elif dst.exists():
    print(f"{dst.name} already exists ({dst.stat().st_size / 1e9:.1f} GB) – skipping.")
else:
    print(f"Compressing {src.name} ({src.stat().st_size / 1e9:.1f} GB) …")
    cctx = zstd.ZstdCompressor(level=3, threads=-1)
    with src.open("rb") as f_in, dst.open("wb") as f_out:
        cctx.copy_stream(f_in, f_out)
    print(f"Done. {src.stat().st_size / 1e9:.1f} GB → {dst.stat().st_size / 1e9:.1f} GB")
    print(f"Upload {dst} instead of the uncompressed .npy to save time.")